오픈소스 NER 모델이 맘에 안들어서 LLM으로 되는지 테스트 중

In [5]:
# !pip install "torch>=2.2" "transformers>=4.51.0" "accelerate>=0.33.0" "sentencepiece" "protobuf" "urllib3<2.4.0,>=1.24.2" --force-reinstall

In [1]:
import os
import sys

# 주피터 노트북 환경에서 __file__이 없으므로, 현재 워킹 디렉토리 기준으로 설정
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
from modeling.models.NER import QwenBasedNER

ner = QwenBasedNER()

/Users/user/miniconda3/envs/dev/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.94s/it]


In [3]:
# 일단 NER로 자연어 쿼리에서 필터링 조건 뽑아서 결과 뽑아주는 기능부터 개발해보자.
query = "김민규가 나오는 진지한 분위기의 로맨틱한 영화 추천해줘"
result = ner.run(query, verbose=True)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🎬 모델 응답:
{
  "actors": ["김민규"],
  "genres": ["로맨스"],
  "years": [],
  "directors": [],
  "movie_titles": [],
  "regions": [],
  "story_keywords": ["진지한"],
  "other_keywords": ["추천"]
}

📊 추출된 정보 (구조화된 데이터):
{
  "actors": [
    "김민규"
  ],
  "genres": [
    "로맨스"
  ],
  "years": [],
  "directors": [],
  "movie_titles": [],
  "regions": [],
  "story_keywords": [
    "진지한"
  ],
  "other_keywords": [
    "추천"
  ]
}

📋 상세 정보:
  배우: 김민규
  장르: 로맨스
  스토리 키워드: 진지한
  기타 키워드: 추천

✅ 완료!


In [5]:
print(result.actors)
print(result.genres)

['김민규']
['로맨스']


In [8]:
from data_scraping.common import Config, DataStorage

config = Config()

config.DATA_DIR = "/Users/user/Desktop/movie-dev/movie_recommendation/data_scraping/data"
storage = DataStorage(config)

infos = storage.load_movie_info()

In [28]:
tmp = """mOPVAe7/키타큐슈 : 영화의 도시/2021/드라마 단편/일본/11분//[('혼다 카츠야', '감독'), ('미츠이시 켄', '주연 | 켄이치')]/"현실이 픽션을 능가하고 있다. 영화의 중요성은 예전만 못해졌고, 이는 영화관 역시 마찬가지다. 영화에 미래란 없다고 생각한다. 마치 사라지지 않으려 고군분투하는 동물 같달까. 어찌 됐든 호랑이는 멸종될 테고 영화관 역시 자취를 감추게 될 것이다. 코로나는 단지 그 시기를 앞당기는 촉매제일 뿐이다." [2023년 제40회 부산국제단편영화제/마티유 카소비츠]/2.8/15/0
"""

a = tmp.split("/")
a

['mOPVAe7',
 '키타큐슈 : 영화의 도시',
 '2021',
 '드라마 단편',
 '일본',
 '11분',
 '',
 "[('혼다 카츠야', '감독'), ('미츠이시 켄', '주연 | 켄이치')]",
 '"현실이 픽션을 능가하고 있다. 영화의 중요성은 예전만 못해졌고, 이는 영화관 역시 마찬가지다. 영화에 미래란 없다고 생각한다. 마치 사라지지 않으려 고군분투하는 동물 같달까. 어찌 됐든 호랑이는 멸종될 테고 영화관 역시 자취를 감추게 될 것이다. 코로나는 단지 그 시기를 앞당기는 촉매제일 뿐이다." [2023년 제40회 부산국제단편영화제',
 '마티유 카소비츠]',
 '2.8',
 '15',
 '0\n']

In [ ]:
def parse_cast_production(cast_str):
    try:
        return cast_str
    except Exception:
        return []


cast_lists = infos["Cast_Production"].apply(parse_cast_production)
cast_lists

"[('김기덕', '감독'), ('이얼', '주연 | 영기'), ('곽지민', '주연 | 여진'), ('한여름', '주연'), ('권현민', '조연 | 센서상'), ('임균호', '조연 | 깔끔 남'), ('정윤수', '조연 | 터프 남'), ('오용', '조연 | 뮤직 남'), ('이종길', '조연 | 행운 남'), ('신택기', '조연 | 자살 남'), ('박정기', '단역'), ('김귀선', '단역'), ('서승원', '단역'), ('유재익', '단역'), ('이수정', '단역'), ('정인기', '단역'), ('전진배', '단역'), ('육세진', '단역'), ('홍혜령', '단역'), ('설한솔', '단역'), ('김재영', '단역'), ('손영순', '단역'), ('박중현', '단역'), ('리다해', '단역')]"

In [11]:
infos

,MovieID,Title,Year,Genre,Country,Runtime,Age,Cast_Production,Synopsis,Avg_Rating,N_Rating,N_Comments
0,m45nEnd,사마리아,2004,드라마,한국,1시간 35분,청불,"[('김기덕', '감독'), ('이얼', '주연 | 영기'), ('곽지민', '주연...","""인도에 바수밀다 라는 창녀가 있었어. 그런데 그 창녀랑 잠만 자고 나면 남자들이 ...",2.8,3.4,500+
1,mWLjGNd,택시 드라이버,1976,드라마 스릴러,미국,1시간 53분,청불,"[('마틴 스콜세지', '감독'), ('로버트 드 니로', '주연 | 트래비스'),...",트래비스(로버트 드 니로)는 베트남전에서 귀환한 후 불면증에 시달리며 사회에 적응하...,4.0,10.8,4000+
2,mOk6BPQ,파울볼,2014,다큐멘터리,한국,1시간 27분,전체,"[('조정래', '감독'), ('김보경', '감독'), ('조진웅', '나레이션')...","한,미,일 3개국 프로야구 선수 출신 최향남, 국내 프로야구 신인왕 출신 김수경 등...",3.3,7355.0,500+
3,mdBzPGd,청설,2009,드라마 로맨스,대만,1시간 49분,전체,"[('청펀펀', '감독'), ('펑위옌', '주연 | 티엔커'), ('진의함', '...",손으로 말하는 ‘양양’과 그녀에게 첫눈에 반한 ‘티엔커’. 마음이 듣고 가슴으로 느...,3.8,16.5,2000+
4,mObJJRO,하녀 라본느,None,None,None,1시간 22분,청불,"[('살바토르 샘페리', '감독'), ('플로랑스 구에린', '주연'), ('트린 ...",None,2.5,36.0,10+
...,...,...,...,...,...,...,...,...,...,...,...,...
70200,mOVvDLg,베를린 함락,1950,전쟁 드라마,소련,2시간 47분,전체,"[('미하일 치아우레리', '감독')]",2차 세계대전의 역사를 당시 소련의 지도자 스탈린의 긍정적인 묘사에 집중하여 그려낸...,3.1,13,0
70201,md6Y9wZ,어느 사진가의 기억,2014,다큐멘터리,한국,1시간 28분,전체,"[('이창민', '감독')]","2011년 5월, 사진가 김영수 선생이 세상을 떠났다. 한국 현대사의 질곡을 겪으며...",3.2,16,0
70202,mO8XQPe,드림쏭2,2021,키즈 애니메이션,"미국, 중국",1시간 30분,전체,"[('마크 밸도', '감독'), ('그레이엄 해밀턴', '성우 | 보디'), ('애...","더 큰 무대로 돌아왔다! 이번엔 월드 투어다! 드림쏭 이후 1년, ‘버디’와 그의 ...",3.0,26,1
70203,mOgEGN5,미란다,,,,1시간 30분,전체,"[('마크 먼든', '감독'), ('크리스티나 리치', '주연 | 미란다'), ('...","본 사이트의 모든 콘텐츠는 왓챠피디아의 자산이며, 사전 동의 없이 복제, 전재, 재...",3.1,19,


성능은 나쁘지 않은데 시간이 15초나 걸리네... 흠 -> 안내문구로 설명 해주자 나중에 할 todo로 사용하게

NER로 뽑은 태그들로 필터링을 한다면, 그 다음엔 뭐 해야 하지?

- 평가한 정보가 있다면 그걸 바탕으로 개인화된 검색 결과 주기
- 없다면 그냥 필터링한 결과 주기
- 스토리 키워드가 제공 된다면 시놉시스랑 시멘틱 서치를 통해 결과 주기